In [5]:
%%capture
!pip install fastapi uvicorn pyngrok nest-asyncio transformers peft accelerate bitsandbytes

In [ ]:
import os
import sys

# 🚨 THE ANTI-OOM FIX: Force PyTorch to prevent memory fragmentation
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

import torch
import nest_asyncio
from pyngrok import ngrok
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
import uvicorn
from transformers import BertTokenizerFast, AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
from kaggle_secrets import UserSecretsClient

# 1. SECURE HUGGING FACE TOKEN INITIALIZATION
try:
    user_secrets = UserSecretsClient()
    os.environ["HF_TOKEN"] = ""
    print("✅ Hugging Face token loaded successfully from Secrets.")
except Exception as e:
    os.environ["HF_TOKEN"] = ""
    print("⚠️ Warning: Using hardcoded HF_TOKEN fallback.")

# 2. MAP KAGGLE PATHS
LOGLLM_CODE_DIR = "/kaggle/input/datasets/avyukthnunna/logllm-code"
sys.path.append(LOGLLM_CODE_DIR)

from customDataset import CustomCollator, CustomDataset
from model import LogLLM
import ast

# 3. SETUP FASTAPI & CORS
app = FastAPI()
nest_asyncio.apply()

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"], 
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

# 4. NGROK AUTHENTICATION (Replace with your token from ngrok.com)
ngrok.set_auth_token("")

listener = ngrok.connect(
    addr=8000,
    domain="joey-obliging-recently.ngrok-free.app"
)

public_url = listener.public_url

print(f"🔥 Live API: {public_url}")

# 5. LOAD BOTH MODELS INTO VRAM
print("Loading Models... (This takes a few minutes)")

# --- LOAD LogLLM (8B) ---
bert_tokenizer = BertTokenizerFast.from_pretrained('bert-base-uncased', do_lower_case=True)
collator = CustomCollator(bert_tokenizer, max_seq_len=128, max_content_len=100)

logllm_model = LogLLM(
    Bert_path='bert-base-uncased',
    Llama_path='meta-llama/Meta-Llama-3-8B', 
    ft_path="/kaggle/input/datasets/avyukthnunna/logllm-checkpoint/ft_model_HDFS",
    is_train_mode=False,
    device="cuda:0" 
)
logllm_model.eval()

# --- LOAD Explanation Model (3B) ---
from transformers import BitsAndBytesConfig # 🚨 NEW IMPORT

BASE_MODEL = "meta-llama/Llama-3.2-3B-Instruct"
ADAPTER_REPO = "kovidritesh/Llama-3.2-3B-FineTuned"

expl_tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

# 🚨 THE FIX: Wrap the 4-bit instructions explicitly so transformers doesn't break
expl_bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

base_expl_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=expl_bnb_config, # Pass the config object here
    device_map="auto" # Automatically shifts this 3B model entirely onto the second GPU (cuda:1)
)
expl_model = PeftModel.from_pretrained(base_expl_model, ADAPTER_REPO)
expl_model.eval()

print("✅ Both models successfully loaded into T4 x2 VRAM!")

HDFS_LOG_PATH = "/kaggle/input/datasets/avyukthnunna/hdfs-dataset/HDFS.log"

with open(HDFS_LOG_PATH, "r") as f:
    HDFS_LOGS = [
        line.strip()
        for line in f
        if line.strip()
    ]

current_log_index = 0


# 6. DEFINE API INPUT STRUCTURE
class LogRequest(BaseModel):
    block_id: str
    semantic_text_sequence: list 
    event_id_sequence: list      
    latency: float

import pandas as pd
from datetime import datetime

@app.get("/live-log")
async def get_live_log():

    global current_log_index

    log_line = HDFS_LOGS[current_log_index]

    current_log_index = (
        current_log_index + 1
    ) % len(HDFS_LOGS)

    return {
        "lvl": "INFO",
        "ts": datetime.now().strftime("%H:%M:%S"),
        "text": log_line
    }
    
# 7. THE CORRECTED PREDICTION ENDPOINT
@app.post("/analyze")
async def analyze_log(req: LogRequest):
    temp_csv_path = "./temp_inference_row.csv"
    try:
        # Step A: Format content exactly like your original pipeline
        formatted_content = " ;-; ".join(req.semantic_text_sequence)
        
        # Write a temporary 1-row CSV to satisfy CustomDataset
        temp_df = pd.DataFrame([{"Content": formatted_content, "Label": 0}])
        temp_df.to_csv(temp_csv_path, index=False)
        
        # Step B: Pass it through your REAL dataset and collator pipeline
        from torch.utils.data import DataLoader
        infer_dataset = CustomDataset(temp_csv_path)
        infer_dataloader = DataLoader(infer_dataset, batch_size=1, shuffle=False, collate_fn=collator)
        
        # Extract the perfectly structured batch tensors
        batch = next(iter(infer_dataloader))
        inputs = {k: v.to(logllm_model.device) for k, v in batch['inputs'].items()}
        seq_positions = batch['seq_positions'] # The real, calculated sequence positions!
        
        # Step C: Run Anomaly Detection
        is_anomaly = False
        with torch.no_grad(), torch.autocast(device_type="cuda", dtype=torch.bfloat16):
            output_tokens = logllm_model(inputs, seq_positions)
            text = logllm_model.Llama_tokenizer.decode(output_tokens[0], skip_special_tokens=True).lower()
            if "anomalous" in text:
                is_anomaly = True
                
        # Clean up the temp file from disk immediately
        if os.path.exists(temp_csv_path):
            os.remove(temp_csv_path)
                
        # Step D: If Normal, return early
        if not is_anomaly:
            return {"block_id": req.block_id, "status": "Normal", "explanation": "System operating as expected. No critical anomalies detected in this block."}
            
        # Step E: If Anomalous, generate explanation with Llama 3B
        prompt = f"Below is an anomalous HDFS log sequence. Explain what went wrong.\n\n### Sequence:\n{req.event_id_sequence}\n\n### Latency:\n{req.latency} sec\n\n### Explanation:\n"
        
        exp_inputs = expl_tokenizer([prompt], return_tensors="pt").to("cuda")
        with torch.no_grad():
            outputs = expl_model.generate(**exp_inputs, max_new_tokens=300, use_cache=True)
            
        full_output = expl_tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]
        explanation = full_output.split("### Explanation:\n")[-1].strip()
        
        return {"block_id": req.block_id, "status": "Anomalous", "explanation": explanation}
        
    except Exception as e:
        if os.path.exists(temp_csv_path):
            os.remove(temp_csv_path)
        raise HTTPException(status_code=500, detail=str(e))

import asyncio
import uvicorn

print("🚀 Starting API Server using the native Jupyter loop...")

# Define the server configuration explicitly
config = uvicorn.Config(
    app, 
    host="0.0.0.0", 
    port=8000, 
    log_level="info",
    loop="asyncio"
)
server = uvicorn.Server(config)

# 🚨 THE FIX: Hook directly into Kaggle's running event loop instead of spawning a new one
await server.serve()


✅ Hugging Face token loaded successfully from Secrets.
🔥 YOUR LIVE API URL: https://bfc9-34-80-210-118.ngrok-free.app                                     
Copy this URL and paste it into your React App.jsx file!
Loading Models... (This takes a few minutes)


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/654 [00:00<?, ?B/s]

...

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

adapter_config.json: 0.00B [00:00, ?B/s]

/usr/local/lib/python3.12/dist-packages/peft/config.py:220: UserWarning: Unexpected keyword arguments ['lora_ga_config', 'use_bdlora'] for class LoraConfig, these are ignored. This probably means that you're loading a configuration file that was saved using a higher version of the library and additional parameters have been introduced since. It is highly recommended to upgrade the PEFT version before continuing (e.g. by running `pip install -U peft`).
  warnings.warn(


adapter_model.safetensors:   0%|          | 0.00/97.3M [00:00<?, ?B/s]

✅ Both models successfully loaded into T4 x2 VRAM!


In [ ]:
import asyncio
import uvicorn

print("🚀 Starting API Server using the native Jupyter loop...")

# Define the server configuration explicitly
config = uvicorn.Config(
    app, 
    host="0.0.0.0", 
    port=8000, 
    log_level="info",
    loop="asyncio"
)
server = uvicorn.Server(config)

# 🚨 THE FIX: Hook directly into Kaggle's running event loop instead of spawning a new one
await server.serve()

INFO:     Started server process [58]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


🚀 Starting API Server using the native Jupyter loop...
INFO:     2401:4900:8f78:9ad9:3468:cd83:200a:483b:0 - "OPTIONS / HTTP/1.1" 200 OK
INFO:     2401:4900:8f78:9ad9:3468:cd83:200a:483b:0 - "OPTIONS / HTTP/1.1" 200 OK
INFO:     2401:4900:8f78:9ad9:3468:cd83:200a:483b:0 - "GET / HTTP/1.1" 404 Not Found
INFO:     2401:4900:8f78:9ad9:3468:cd83:200a:483b:0 - "GET / HTTP/1.1" 404 Not Found
INFO:     2401:4900:8f78:9ad9:3468:cd83:200a:483b:0 - "GET / HTTP/1.1" 404 Not Found
INFO:     2401:4900:8f78:9ad9:3468:cd83:200a:483b:0 - "GET / HTTP/1.1" 404 Not Found
INFO:     2401:4900:8f78:9ad9:3468:cd83:200a:483b:0 - "GET / HTTP/1.1" 404 Not Found
INFO:     2401:4900:8f78:9ad9:3468:cd83:200a:483b:0 - "GET / HTTP/1.1" 404 Not Found
INFO:     2401:4900:8f78:9ad9:3468:cd83:200a:483b:0 - "GET / HTTP/1.1" 404 Not Found
INFO:     2401:4900:8f78:9ad9:3468:cd83:200a:483b:0 - "GET / HTTP/1.1" 404 Not Found
INFO:     2401:4900:8f78:9ad9:3468:cd83:200a:483b:0 - "GET / HTTP/1.1" 404 Not Found
INFO:     2401:4

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


INFO:     2401:4900:8f78:9ad9:3468:cd83:200a:483b:0 - "POST /analyze HTTP/1.1" 200 OK
INFO:     2401:4900:8f78:9ad9:3468:cd83:200a:483b:0 - "GET / HTTP/1.1" 404 Not Found
INFO:     2401:4900:8f78:9ad9:3468:cd83:200a:483b:0 - "GET / HTTP/1.1" 404 Not Found
INFO:     2401:4900:8f78:9ad9:3468:cd83:200a:483b:0 - "GET / HTTP/1.1" 404 Not Found
INFO:     2401:4900:8f78:9ad9:3468:cd83:200a:483b:0 - "GET / HTTP/1.1" 404 Not Found
INFO:     2401:4900:8f78:9ad9:3468:cd83:200a:483b:0 - "GET / HTTP/1.1" 404 Not Found
INFO:     2401:4900:8f78:9ad9:3468:cd83:200a:483b:0 - "GET / HTTP/1.1" 404 Not Found
INFO:     2401:4900:8f78:9ad9:3468:cd83:200a:483b:0 - "GET / HTTP/1.1" 404 Not Found
INFO:     2401:4900:8f78:9ad9:3468:cd83:200a:483b:0 - "GET / HTTP/1.1" 404 Not Found
INFO:     2401:4900:8f78:9ad9:3468:cd83:200a:483b:0 - "GET / HTTP/1.1" 404 Not Found
INFO:     2401:4900:8f78:9ad9:3468:cd83:200a:483b:0 - "OPTIONS / HTTP/1.1" 200 OK
INFO:     2401:4900:8f78:9ad9:3468:cd83:200a:483b:0 - "GET / HTTP/1

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


INFO:     2401:4900:8f78:9ad9:3468:cd83:200a:483b:0 - "POST /analyze HTTP/1.1" 200 OK
INFO:     2401:4900:8f78:9ad9:3468:cd83:200a:483b:0 - "GET / HTTP/1.1" 404 Not Found
INFO:     2401:4900:8f78:9ad9:3468:cd83:200a:483b:0 - "GET / HTTP/1.1" 404 Not Found
INFO:     2401:4900:8f78:9ad9:3468:cd83:200a:483b:0 - "GET / HTTP/1.1" 404 Not Found
INFO:     2401:4900:8f78:9ad9:3468:cd83:200a:483b:0 - "GET / HTTP/1.1" 404 Not Found
INFO:     2401:4900:8f78:9ad9:3468:cd83:200a:483b:0 - "GET / HTTP/1.1" 404 Not Found
INFO:     2401:4900:8f78:9ad9:3468:cd83:200a:483b:0 - "GET / HTTP/1.1" 404 Not Found
INFO:     2401:4900:8f78:9ad9:3468:cd83:200a:483b:0 - "GET / HTTP/1.1" 404 Not Found
INFO:     2401:4900:8f78:9ad9:3468:cd83:200a:483b:0 - "OPTIONS / HTTP/1.1" 200 OK
INFO:     2401:4900:8f78:9ad9:3468:cd83:200a:483b:0 - "GET / HTTP/1.1" 404 Not Found
INFO:     2401:4900:8f78:9ad9:3468:cd83:200a:483b:0 - "GET / HTTP/1.1" 404 Not Found
INFO:     2401:4900:8f78:9ad9:3468:cd83:200a:483b:0 - "GET / HTTP/1

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


INFO:     2401:4900:8f78:9ad9:3468:cd83:200a:483b:0 - "POST /analyze HTTP/1.1" 200 OK
INFO:     2401:4900:8f78:9ad9:3468:cd83:200a:483b:0 - "GET / HTTP/1.1" 404 Not Found
INFO:     2401:4900:8f78:9ad9:3468:cd83:200a:483b:0 - "GET / HTTP/1.1" 404 Not Found
INFO:     2401:4900:8f78:9ad9:3468:cd83:200a:483b:0 - "GET / HTTP/1.1" 404 Not Found
INFO:     2401:4900:8f78:9ad9:3468:cd83:200a:483b:0 - "GET / HTTP/1.1" 404 Not Found
INFO:     2401:4900:8f78:9ad9:3468:cd83:200a:483b:0 - "OPTIONS /live-log HTTP/1.1" 200 OK
INFO:     2401:4900:8f78:9ad9:3468:cd83:200a:483b:0 - "GET /live-log HTTP/1.1" 200 OK
INFO:     2401:4900:8f78:9ad9:3468:cd83:200a:483b:0 - "GET /live-log HTTP/1.1" 200 OK
INFO:     2401:4900:8f78:9ad9:3468:cd83:200a:483b:0 - "GET /live-log HTTP/1.1" 200 OK
INFO:     2401:4900:8f78:9ad9:3468:cd83:200a:483b:0 - "GET /live-log HTTP/1.1" 200 OK
INFO:     2401:4900:8f78:9ad9:3468:cd83:200a:483b:0 - "GET /live-log HTTP/1.1" 200 OK
INFO:     2401:4900:8f78:9ad9:3468:cd83:200a:483b:0 - 

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


INFO:     2401:4900:8f78:9ad9:3468:cd83:200a:483b:0 - "POST /analyze HTTP/1.1" 200 OK
INFO:     2401:4900:8f78:9ad9:3468:cd83:200a:483b:0 - "GET / HTTP/1.1" 404 Not Found
INFO:     2401:4900:8f78:9ad9:3468:cd83:200a:483b:0 - "GET / HTTP/1.1" 404 Not Found
INFO:     2401:4900:8f78:9ad9:3468:cd83:200a:483b:0 - "GET / HTTP/1.1" 404 Not Found
INFO:     2401:4900:8f78:9ad9:3468:cd83:200a:483b:0 - "GET / HTTP/1.1" 404 Not Found
INFO:     2401:4900:8f78:9ad9:3468:cd83:200a:483b:0 - "GET / HTTP/1.1" 404 Not Found
INFO:     2401:4900:8f78:9ad9:3468:cd83:200a:483b:0 - "GET / HTTP/1.1" 404 Not Found
INFO:     2401:4900:8f78:9ad9:3468:cd83:200a:483b:0 - "GET / HTTP/1.1" 404 Not Found
INFO:     2401:4900:8f78:9ad9:3468:cd83:200a:483b:0 - "GET / HTTP/1.1" 404 Not Found
INFO:     2401:4900:8f78:9ad9:3468:cd83:200a:483b:0 - "GET / HTTP/1.1" 404 Not Found
INFO:     2401:4900:8f78:9ad9:3468:cd83:200a:483b:0 - "GET / HTTP/1.1" 404 Not Found
INFO:     2401:4900:8f78:9ad9:3468:cd83:200a:483b:0 - "GET / HTT